# Diseño de un Pipeline ETL en Python 🐍🔧

¡Hola, equipo! 👋 En esta sesión vamos a construir nuestro primer **pipeline de ETL (Extract, Transform, Load)** en Python. Es una de las tareas más comunes y fundamentales en el mundo de los datos.

> 🧠 **Analogía:** Piensa en un ETL como el proceso de **cocinar**. Primero, **extraes** los ingredientes (datos) del supermercado (API, base de datos, etc.). Luego, los **transformas**: los lavas, cortas y preparas. Finalmente, los **cargas** en un plato para servirlos (una base de datos, un dashboard).

## Objetivos del Webinar 🎯

1.  **Explicar qué es ETL** (y cuándo usarlo).
2.  **Extraer (E)** datos desde el API de OpenAQ.
3.  **Transformar (T)** aplicando limpieza, cambios de tipo y validaciones.
4.  **Crear tablas de resumen** (nuestra capa de "oro" o `gold_data`).
5.  **Cargar (L)** los datos en una base de datos SQLite.
6.  **Ensamblar un pipeline** simple y reutilizable con funciones.

## 🏛️ ¿Qué es ETL y por qué es tan importante?

**ETL** significa **Extract, Transform, Load** (Extraer, Transformar, Cargar). Es el proceso que usamos para mover datos desde una o varias fuentes, limpiarlos, darles una estructura útil y almacenarlos en un destino final, como una base de datos o un Data Warehouse.

-   **Extract (Extraer):** Obtener los datos desde su origen.
-   **Transform (Transformar):** Limpiar, validar, enriquecer y modelar los datos. ¡Aquí es donde ocurre la magia! ✨
-   **Load (Cargar):** Guardar los datos transformados en su destino final.

Usamos ETL para consolidar datos, prepararlos para análisis, alimentar dashboards o para cualquier tarea que requiera datos limpios y estructurados.

## 1. Extract: Extrayendo datos del API de OpenAQ 🌍

Vamos a usar el API de [OpenAQ](https://openaq.org/) para obtener datos de calidad del aire. Ya vimos cómo conectarnos en sesiones anteriores, ¡así que vamos a aplicar lo aprendido!

Nuestro objetivo será extraer las mediciones de `pm25` para un país específico en un rango de fechas.

In [71]:
import requests
import pandas as pd

In [72]:
API_KEY="81212e4dcaa02641d5fefedc6c1e43efd2bf732e7c13d2cbd84096d358330792"

In [73]:
def get_country_data():
    url = "https://api.openaq.org/v3/countries"
    headers = {
        "X-API-Key": API_KEY
    }

    params={'limit':1000}
    response = requests.get(url, headers=headers,params=params)
    response.raise_for_status()
    return response.json()


def get_locations_data_by_country_ids(country_ids):
    url = "https://api.openaq.org/v3/locations"


    headers = {
        "X-API-Key": API_KEY
    }

    params={'limit':1000,'countries_id':country_ids}
    response = requests.get(url, headers=headers,params=params)
    response.raise_for_status()

    data = response.json()

    return data


def process_locations_sensor_data(location_data):
    locations_sensors=pd.DataFrame(location_data['results'])
    locations_sensors=locations_sensors.explode('sensors')
    sensor_data=pd.json_normalize(locations_sensors['sensors']).reset_index(drop=True)
    locations_sensors=locations_sensors[['id','name','locality']].reset_index(drop=True)
    locations_sensors.rename(columns={'id':'id_location','name':'name_location'},inplace=True)
    locations_sensors = pd.concat([locations_sensors,sensor_data],axis=1)
    return locations_sensors


def get_daily_measurements_by_sensor_id(sensor_id,params):

    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily"
    headers = {
        "X-API-Key": API_KEY
    }
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    return response.json()['results']



### Extracción de los datos 

In [74]:
data= get_country_data()
countries=pd.DataFrame(data['results'])
countries.query('code == "MX"')

,id,code,name,datetimeFirst,datetimeLast,parameters
123,157,MX,Mexico,2016-03-06T20:00:00Z,2026-03-26T22:00:00Z,"[{'id': 1, 'name': 'pm10', 'units': 'µg/m³', '..."


In [75]:
#Params
CODE_COUNTRY=157

In [76]:
locations_data= get_locations_data_by_country_ids(157)
locations_data = process_locations_sensor_data(locations_data)

In [77]:
locations_data

,id_location,name_location,locality,id,name,parameter.id,parameter.name,parameter.units,parameter.displayName
0,309,Obispado,None,522,o3 ppm,10,o3,ppm,O₃
1,309,Obispado,None,1027,pm10 µg/m³,1,pm10,µg/m³,PM10
2,338,Estaci�n Hospital Ge,GUANAJUATO,3761,co ppm,8,co,ppm,CO
3,338,Estaci�n Hospital Ge,GUANAJUATO,3760,o3 ppm,10,o3,ppm,O₃
4,338,Estaci�n Hospital Ge,GUANAJUATO,3756,pm10 µg/m³,1,pm10,µg/m³,PM10
...,...,...,...,...,...,...,...,...,...
1737,6271741,San Jose del cabo,None,15682375,pm1 µg/m³,19,pm1,µg/m³,PM1
1738,6271741,San Jose del cabo,None,15682376,pm25 µg/m³,2,pm25,µg/m³,PM2.5
1739,6271741,San Jose del cabo,None,15682377,relativehumidity %,98,relativehumidity,%,RH
1740,6271741,San Jose del cabo,None,15682378,temperature c,100,temperature,c,Temperature (C)


In [78]:
locations_data.query('locality == "GUANAJUATO" and name == "pm25 µg/m³"')

,id_location,name_location,locality,id,name,parameter.id,parameter.name,parameter.units,parameter.displayName
5,338,Estaci�n Hospital Ge,GUANAJUATO,581,pm25 µg/m³,2,pm25,µg/m³,PM2.5
15,359,Estaci�n Bomberos,GUANAJUATO,615,pm25 µg/m³,2,pm25,µg/m³,PM2.5
24,413,Estaci�n Te�dula,GUANAJUATO,5079213,pm25 µg/m³,2,pm25,µg/m³,PM2.5
72,676,Estaci�n CICEG,GUANAJUATO,1926298,pm25 µg/m³,2,pm25,µg/m³,PM2.5
81,684,Estaci�n Seguridad P,GUANAJUATO,1175,pm25 µg/m³,2,pm25,µg/m³,PM2.5
226,2094,Estaci�n San Juanico,GUANAJUATO,3748,pm25 µg/m³,2,pm25,µg/m³,PM2.5
231,2097,Estaci�n Facultad de,GUANAJUATO,3762,pm25 µg/m³,2,pm25,µg/m³,PM2.5
236,2098,Estaci�n DIF,GUANAJUATO,3767,pm25 µg/m³,2,pm25,µg/m³,PM2.5
252,6368,Estaci�n Guanajuato,GUANAJUATO,17827,pm25 µg/m³,2,pm25,µg/m³,PM2.5
254,6919,Estaci�n San Luis de,GUANAJUATO,19866,pm25 µg/m³,2,pm25,µg/m³,PM2.5


In [79]:
ids_selection=locations_data.query('locality == "GUANAJUATO" and name == "co ppm"')['id'].to_list()

In [80]:
locations_data[['name_location','id_location','locality','id']]

,name_location,id_location,locality,id
0,Obispado,309,None,522
1,Obispado,309,None,1027
2,Estaci�n Hospital Ge,338,GUANAJUATO,3761
3,Estaci�n Hospital Ge,338,GUANAJUATO,3760
4,Estaci�n Hospital Ge,338,GUANAJUATO,3756
...,...,...,...,...
1737,San Jose del cabo,6271741,None,15682375
1738,San Jose del cabo,6271741,None,15682376
1739,San Jose del cabo,6271741,None,15682377
1740,San Jose del cabo,6271741,None,15682378


In [81]:
raw_data =[]
params = {
    "datetime_from": "2025-01-01T00:00:00Z",
    "datetime_to": "2025-01-31T23:59:59Z",
    "limit": 31
}
for sensor_id in ids_selection:
    try:
        current_data=get_daily_measurements_by_sensor_id(sensor_id,params)
        raw_data.append({'sensor_id':sensor_id,'data':current_data})
    except Exception as ee:
        print("Failed")
        print (ee)
        pass


## 2. Transform: Limpieza y modelado de datos 🧹

Los datos crudos rara vez son perfectos. La etapa de transformación es crucial para asegurar la calidad.

Nuestras tareas serán:
- Convertir la lista de diccionarios a un DataFrame de pandas.
- Seleccionar solo las columnas que nos interesan.
- Convertir la fecha a un formato `datetime`.
- Desanidar la información .
- Renombrar columnas para que sean más claras.
- Validar que los valores de `pm25` no sean negativos.

In [82]:
def cleaning_data_sensor(raw_data):
    df=pd.DataFrame(raw_data) 
    df=df.explode('data')
    df = pd.concat([df[['sensor_id']].reset_index(drop=True),pd.json_normalize(df['data']).reset_index(drop=True)],axis=1)
    df['date_at']=pd.to_datetime(df['period.datetimeFrom.local'])
    df.rename(columns={'parameter.name':'parameter'},inplace=True)
    df=df[['date_at','sensor_id','parameter','value']]
    return df[df['value']>=0]


In [83]:
locations_data=locations_data[['id','name_location','locality']].rename(columns={'id':'sensor_id'})
cleaning_measures_data=cleaning_data_sensor(raw_data)

In [84]:
df_silver=locations_data.merge(cleaning_measures_data)

In [87]:
df_silver

,sensor_id,name_location,locality,date_at,parameter,value
0,3761,Estaci�n Hospital Ge,GUANAJUATO,2025-01-01 00:00:00-06:00,co,2.300
1,3761,Estaci�n Hospital Ge,GUANAJUATO,2025-01-02 00:00:00-06:00,co,2.300
2,3761,Estaci�n Hospital Ge,GUANAJUATO,2025-01-03 00:00:00-06:00,co,2.250
3,3761,Estaci�n Hospital Ge,GUANAJUATO,2025-01-04 00:00:00-06:00,co,2.340
4,3761,Estaci�n Hospital Ge,GUANAJUATO,2025-01-05 00:00:00-06:00,co,2.660
...,...,...,...,...,...,...
239,3771,Estaci�n DIF,GUANAJUATO,2025-01-27 00:00:00-06:00,co,1.270
240,3771,Estaci�n DIF,GUANAJUATO,2025-01-28 00:00:00-06:00,co,0.963
241,3771,Estaci�n DIF,GUANAJUATO,2025-01-29 00:00:00-06:00,co,0.838
242,3771,Estaci�n DIF,GUANAJUATO,2025-01-30 00:00:00-06:00,co,0.725


> En la industria, a esta capa de datos limpios y estructurados a menudo se le llama **"Silver Layer"** (Capa de Plata).

## 3. Creando la capa "Gold": Tablas de resumen 🏆

La capa "Gold" (Oro) contiene datos agregados y listos para el negocio. Son las tablas que un analista o un modelo de Machine Learning consumiría directamente.

Vamos a crear una tabla que resuma el promedio diario de `pm25` por ciudad.

In [88]:
def create_gold_data(df):
    """
    Crea una tabla de resumen (capa Gold) con el promedio diario de pm25 por ciudad.
    """
    if df.empty:
        return pd.DataFrame()

    print("🏆 Creando la tabla de resumen (Gold)...")
    
    df['date'] = df['date_at'].dt.date
    df=df.rename(columns={'locality':'city'})
    
    df_gold = df.groupby(['city', 'date']).agg(
        avg_pm25=('value', 'mean'),
        max_pm25=('value', 'max'),
        min_pm25=('value', 'min'),
        num_measurements=('value', 'count')
    ).reset_index()
    
    df_gold['avg_pm25'] = df_gold['avg_pm25'].round(2)
    
    print("✨ ¡Tabla Gold creada exitosamente!")
    return df_gold

In [89]:
# Creemos nuestra tabla de oro
df_gold = create_gold_data(df_silver.copy()) # Usamos .copy() para evitar warnings
if not df_gold.empty:
    print(df_gold.head())

🏆 Creando la tabla de resumen (Gold)...
✨ ¡Tabla Gold creada exitosamente!
         city        date  avg_pm25  max_pm25  min_pm25  num_measurements
0  GUANAJUATO  2025-01-01      1.39      2.60     0.400                 8
1  GUANAJUATO  2025-01-02      1.42      2.35     0.421                 8
2  GUANAJUATO  2025-01-03      1.34      2.25     0.417                 8
3  GUANAJUATO  2025-01-04      1.44      2.40     0.425                 8
4  GUANAJUATO  2025-01-05      1.52      2.66     0.450                 8


## 4. Load: Cargando los datos a una Base de Datos SQLite 💾

¡Es hora de guardar nuestro trabajo! Usaremos **SQLite**, una base de datos súper ligera que guarda todo en un solo archivo. Es perfecta para proyectos como este.

Pandas hace que este proceso sea increíblemente fácil con el método `.to_sql()`.

In [ ]:
import sqlite3

def load_to_sqlite(tables, db_name='openaq.db'):
    """
    Carga un diccionario de DataFrames a tablas en una base de datos SQLite.
    - tables: {'nombre_tabla': DataFrame}
    """
    print(f"💾 Cargando datos a la base de datos '{db_name}'...")
    
    try:
        conn = sqlite3.connect(db_name)
        
        for table_name, df in tables.items():
            if not df.empty:
                df.to_sql(table_name, conn, if_exists='replace', index=False)
                print(f"  - Tabla '{table_name}' cargada con {len(df)} filas.")
        
        conn.close()
        print("✅ Carga finalizada.")
        
    except sqlite3.Error as e:
        print(f"🔥 Error al cargar a SQLite: {e}")

In [ ]:
# Preparamos el diccionario de tablas y ejecutamos la carga
tables_to_load = {
    'raw_measurements': cleaning_measures_data, # Guardamos también la data cruda por si acaso
    'silver_measurements': df_silver,
    'gold_daily_summary': df_gold
}

load_to_sqlite(tables_to_load)

### Verificación

¿Cómo sabemos que funcionó? ¡Leamos los datos de vuelta desde la base de datos!

In [ ]:
def verify_load(db_name='openaq.db'):
    conn = sqlite3.connect(db_name)
    
    try:
        print("\n🔍 Verificando datos en la base de datos:")
        # Leemos los nombres de las tablas
        tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
        print("Tablas encontradas:", tables['name'].tolist())

        # Leemos algunas filas de la tabla gold
        df_check = pd.read_sql("SELECT * FROM gold_daily_summary LIMIT 5", conn)
        print("\nPrimeras 5 filas de 'gold_daily_summary':")
        print(df_check)
        
    finally:
        conn.close()

verify_load()

## 5. ¡Juntando todo en un Pipeline! 🚀

Hemos creado funciones para cada paso. Ahora, podemos unirlas en un solo script que ejecute todo el proceso de forma ordenada.

In [ ]:
def run_openaq_etl_pipeline(country_code='MX', date_from='2024-01-01', date_to='2024-01-02'):
    """
    Ejecuta el pipeline ETL completo para OpenAQ.
    """
    print("=============================================")
    print(f"🚀 INICIANDO PIPELINE ETL PARA {country_code} 🚀")
    print("=============================================")
    
    # 1. Extract
    raw_data = extract_measurements(country_code, date_from, date_to)
    
    # 2. Transform
    df_silver = transform_data(raw_data)
    
    # 3. Create Gold Data
    df_gold = create_gold_data(df_silver.copy())
    
    # 4. Load
    if raw_data:
        df_raw = pd.DataFrame(raw_data) # Necesitamos el DF crudo para cargarlo
        tables_to_load = {
            'raw_measurements': df_raw,
            'silver_measurements': df_silver,
            'gold_daily_summary': df_gold
        }
        load_to_sqlite(tables_to_load)
        
        # 5. Verify
        verify_load()
    
    print("\n🎉 ¡Pipeline finalizado exitosamente! 🎉")

In [ ]:
# ¡Ejecutemos todo el flujo con una sola llamada!
run_openaq_etl_pipeline(country_code='CL', date_from='2024-03-01', date_to='2024-03-02')

¡Y ahí lo tienes! Un pipeline de ETL funcional. Este es el pilar para construir sistemas de datos robustos y automatizados.

## Conclusión y Próximos Pasos 🏁

Hoy aprendimos a:
- Estructurar un proceso ETL con funciones claras.
- Manejar datos desde una API hasta una base de datos.
- Diferenciar entre capas de datos (Raw, Silver, Gold).
- Usar SQLite para almacenar nuestros resultados de forma sencilla.

**Para seguir aprendiendo:**
- **Mejora el pipeline:** Añade manejo de errores más robusto, logs para cada paso o un sistema de configuración.
- **Automatízalo:** Investiga cómo programar la ejecución de este script una vez al día (¡hola, `cron` o Airflow!).
- **Expande las transformaciones:** ¿Qué otras métricas podrías calcular en la capa Gold?

¡Excelente trabajo, equipo! 💪✨